# People Analytics — Turnover de Colaboradores
## Notebook 1: Quem são os colaboradores?

**Dataset:** IBM HR Analytics Employee Attrition & Performance  
**Fonte:** [Kaggle](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset)  
**Objetivo:** Entender o perfil demográfico e profissional dos 1.470 colaboradores antes de analisar o turnover.

---

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [2]:
df = pd.read_csv('../data/raw/ibm_hr_attrition.csv')

print(f'Dimensões: {df.shape[0]} colaboradores x {df.shape[1]} variáveis')
print(f'Valores nulos: {df.isnull().sum().sum()}')
df.head(3)

Dimensões: 1470 colaboradores x 35 variáveis
Valores nulos: 0


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0


## 1. Visão Geral do Dataset

In [3]:
# Colunas constantes — não agregam valor analítico
colunas_constantes = [col for col in df.columns if df[col].nunique() == 1]
print('Colunas constantes (serão ignoradas):', colunas_constantes)

df_analise = df.drop(columns=colunas_constantes + ['EmployeeNumber'])
print(f'\nVariáveis para análise: {df_analise.shape[1]}')

Colunas constantes (serão ignoradas): ['EmployeeCount', 'Over18', 'StandardHours']

Variáveis para análise: 31


In [4]:
# Taxa de turnover geral
total = len(df)
saidas = (df['Attrition'] == 'Yes').sum()
taxa = saidas / total * 100

fig = go.Figure(go.Indicator(
    mode='gauge+number',
    value=taxa,
    number={'suffix': '%', 'font': {'size': 48}},
    title={'text': 'Taxa de Turnover Geral', 'font': {'size': 20}},
    gauge={
        'axis': {'range': [0, 40]},
        'bar': {'color': '#E63946'},
        'steps': [
            {'range': [0, 10], 'color': '#A8DADC'},
            {'range': [10, 20], 'color': '#F4A261'},
            {'range': [20, 40], 'color': '#FFB3B3'}
        ],
        'threshold': {'line': {'color': 'black', 'width': 3}, 'value': 15}
    }
))
fig.update_layout(height=350)
fig.show()

print(f'{saidas} colaboradores saíram de um total de {total} ({taxa:.1f}%)')
print('Referência: taxa saudável de mercado é entre 5% e 15%')

237 colaboradores saíram de um total de 1470 (16.1%)
Referência: taxa saudável de mercado é entre 5% e 15%


## 2. Perfil Demográfico

In [5]:
# Distribuição de idade
fig = px.histogram(
    df, x='Age', color='Attrition',
    color_discrete_map={'Yes': '#E63946', 'No': '#457B9D'},
    barmode='overlay', opacity=0.75,
    title='Distribuição de Idade por Status de Saída',
    labels={'Age': 'Idade', 'Attrition': 'Saiu?', 'count': 'Colaboradores'},
    nbins=30
)
fig.update_layout(legend_title='Saiu da empresa?')
fig.show()

print('Idade média — ficou:', df[df['Attrition']=='No']['Age'].mean().round(1))
print('Idade média — saiu:', df[df['Attrition']=='Yes']['Age'].mean().round(1))

Idade média — ficou: 37.6
Idade média — saiu: 33.6


In [6]:
# Gênero e estado civil
fig = make_subplots(rows=1, cols=2, specs=[[{'type':'pie'}, {'type':'pie'}]])

genero = df['Gender'].value_counts()
civil = df['MaritalStatus'].value_counts()

fig.add_trace(go.Pie(
    labels=genero.index, values=genero.values,
    name='Gênero', marker_colors=['#457B9D', '#E63946']
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=civil.index, values=civil.values,
    name='Estado Civil', marker_colors=['#A8DADC', '#F4A261', '#E63946']
), row=1, col=2)

fig.update_layout(
    title='Composição da Força de Trabalho: Gênero e Estado Civil',
    annotations=[
        dict(text='Gênero', x=0.18, y=0.5, font_size=14, showarrow=False),
        dict(text='Estado Civil', x=0.82, y=0.5, font_size=14, showarrow=False)
    ]
)
fig.show()

## 3. Perfil Profissional

In [7]:
# Headcount por departamento e cargo
headcount_dept = df['Department'].value_counts().reset_index()
headcount_dept.columns = ['departamento', 'total']

fig = px.bar(
    headcount_dept, x='departamento', y='total',
    color='departamento',
    color_discrete_sequence=['#457B9D', '#E63946', '#F4A261'],
    title='Headcount por Departamento',
    labels={'departamento': 'Departamento', 'total': 'Número de Colaboradores'},
    text='total'
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

In [8]:
# Turnover por departamento
turnover_dept = df.groupby('Department')['Attrition'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
turnover_dept.columns = ['departamento', 'taxa_turnover']
turnover_dept = turnover_dept.sort_values('taxa_turnover', ascending=True)

fig = px.bar(
    turnover_dept, x='taxa_turnover', y='departamento',
    orientation='h',
    color='taxa_turnover',
    color_continuous_scale=['#A8DADC', '#E63946'],
    title='Taxa de Turnover por Departamento (%)',
    labels={'taxa_turnover': 'Taxa de Turnover (%)', 'departamento': ''},
    text=turnover_dept['taxa_turnover'].round(1).astype(str) + '%'
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [9]:
# Distribuição salarial por departamento
fig = px.box(
    df, x='Department', y='MonthlyIncome',
    color='Attrition',
    color_discrete_map={'Yes': '#E63946', 'No': '#457B9D'},
    title='Salário Mensal por Departamento e Status de Saída',
    labels={
        'Department': 'Departamento',
        'MonthlyIncome': 'Salário Mensal (USD)',
        'Attrition': 'Saiu?'
    }
)
fig.show()

## 4. Tempo de Casa e Nível de Cargo

In [10]:
# Tempo de empresa por status de saída
fig = px.violin(
    df, x='Attrition', y='YearsAtCompany',
    color='Attrition',
    color_discrete_map={'Yes': '#E63946', 'No': '#457B9D'},
    box=True, points='outliers',
    title='Tempo de Empresa: Quem fica vs quem sai',
    labels={
        'Attrition': 'Saiu da empresa?',
        'YearsAtCompany': 'Anos na empresa'
    }
)
fig.update_xaxes(ticktext=['Não', 'Sim'], tickvals=['No', 'Yes'])
fig.show()

print('Tempo médio — ficou:', df[df['Attrition']=='No']['YearsAtCompany'].mean().round(1), 'anos')
print('Tempo médio — saiu:', df[df['Attrition']=='Yes']['YearsAtCompany'].mean().round(1), 'anos')

Tempo médio — ficou: 7.4 anos
Tempo médio — saiu: 5.1 anos


In [11]:
# Nível de cargo e turnover
nivel_labels = {1: 'Júnior', 2: 'Pleno', 3: 'Sênior', 4: 'Especialista', 5: 'Diretor'}
df['nivel_cargo'] = df['JobLevel'].map(nivel_labels)

turnover_nivel = df.groupby('nivel_cargo')['Attrition'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
).reset_index()
turnover_nivel.columns = ['nivel', 'taxa']

ordem = ['Júnior', 'Pleno', 'Sênior', 'Especialista', 'Diretor']
turnover_nivel['nivel'] = pd.Categorical(turnover_nivel['nivel'], categories=ordem, ordered=True)
turnover_nivel = turnover_nivel.sort_values('nivel')

fig = px.bar(
    turnover_nivel, x='nivel', y='taxa',
    color='taxa',
    color_continuous_scale=['#A8DADC', '#E63946'],
    title='Taxa de Turnover por Nível de Cargo (%)',
    labels={'nivel': 'Nível de Cargo', 'taxa': 'Taxa de Turnover (%)'},
    text=turnover_nivel['taxa'].round(1).astype(str) + '%'
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 5. Resumo — O que o perfil nos diz

| Dimensão | Insight |
|---|---|
| **Taxa geral** | 16,1% de turnover — acima do limite saudável de 15% |
| **Idade** | Colaboradores mais jovens saem mais — concentração abaixo dos 35 anos |
| **Departamento** | Sales tem a maior taxa de turnover; R&D a menor |
| **Salário** | Quem sai tende a ganhar menos dentro do mesmo departamento |
| **Tempo de casa** | Quem sai tem em média 5 anos a menos de empresa |
| **Nível de cargo** | Turnover é crítico no nível Júnior e cai progressivamente com a senioridade |

> **Próximo passo:** No Notebook 2, vamos aprofundar os fatores que mais influenciam a decisão de sair — satisfação, horas extras, equilíbrio trabalho-vida e promoções.